In [5]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load data
df = pd.read_csv("clean_data.csv")

# 2. Features and target
y = df["severity4"]
groups = df["location_group"]                      # only used for splitting
X = df.drop(columns=["severity4", "location_group"])
X = pd.get_dummies(X, columns=["region", "data_provider"], dtype=int)

# 3. Train/test split (same location never in both sets)
train_idx, test_idx = next(GroupShuffleSplit(test_size=0.2, random_state=42).split(X, y, groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# 4. Scale numeric columns (fit on train only)
num_cols = df.drop(columns=["severity4", "location_group", "region", "data_provider"]).columns
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# 5. Train model
model = LogisticRegression(max_iter=5000, class_weight="balanced")
model.fit(X_train, y_train)

# 6. Evaluate
pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

Accuracy: 0.595
              precision    recall  f1-score   support

           1       0.66      0.74      0.70       264
           2       0.28      0.20      0.24       161
           3       0.51      0.61      0.56       199
           4       0.93      0.75      0.83       137

    accuracy                           0.60       761
   macro avg       0.60      0.58      0.58       761
weighted avg       0.59      0.60      0.59       761

[[195  39  29   1]
 [ 69  33  59   0]
 [ 31  39 122   7]
 [  2   5  27 103]]
